# Comparación final: v1 con 200 casos frente a v1 con 1000 casos

**Consideración metodológica.** A diferencia de las comparaciones anteriores del trabajo, aquí no es posible emplear Wilcoxon ni t-test pareado. Las comparaciones previas (baseline, v1, v2 y Optuna-Best) se realizaron sobre los mismos 40 casos de validación del fold 0 de `Dataset501_ImageCAS`, lo que constituye una comparación pareada válida (mismo paciente, dos modelos).

En este caso, el modelo de 1000 casos se valida sobre el fold 0 de `Dataset503_ImageCAS1000`, cuyo conjunto de validación es **distinto y mayor** (≈ 200 casos frente a 40), de modo que los pacientes no coinciden. Por ello se emplean tests para **muestras independientes**:
- **Mann-Whitney U**, equivalente no paramétrico de Wilcoxon para grupos independientes.
- **t-test independiente** (`ttest_ind`, en lugar de `ttest_rel`).

No se realiza ningún entrenamiento: el notebook carga los CSV caso a caso generados previamente para v1 con 200 casos y para el entrenamiento definitivo de v1 con 1000 casos.


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
!python -m pip install -q pandas scipy

In [3]:
import os
from pathlib import Path
import pandas as pd
from scipy import stats
import numpy as np

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

CSV_200 = BASE / "comparacion_metricas_completo.csv"          # del notebook 19/23, incluye v1_lam02
CSV_1000 = BASE / "v1_1000casos_100epochs_metricas_por_caso.csv"  # del notebook 28

print("CSV 200 casos existe:", CSV_200.exists())
print("CSV 1000 casos existe:", CSV_1000.exists())

CSV 200 casos existe: True
CSV 1000 casos existe: True


## 1. Carga de ambos conjuntos de resultados


In [4]:
df_200 = pd.read_csv(CSV_200)
df_200_v1 = df_200[df_200["modelo"] == "v1_lam02"].copy()
df_200_v1["dataset"] = "200 casos"

df_1000 = pd.read_csv(CSV_1000)
df_1000["dataset"] = "1000 casos"

print(f"v1 (200 casos): {len(df_200_v1)} casos de validacion")
print(f"v1 (1000 casos): {len(df_1000)} casos de validacion")

print("\nColumnas 200 casos:", df_200_v1.columns.tolist())
print("Columnas 1000 casos:", df_1000.columns.tolist())

v1 (200 casos): 40 casos de validacion
v1 (1000 casos): 200 casos de validacion

Columnas 200 casos: ['dice', 'cldice', 'n_componentes', 'cobertura_principal', 'hd95_crudo_mm', 'hd95_principal_mm', 'hd95_filtrado_mm', 'modelo', 'case_id', 'dataset']
Columnas 1000 casos: ['dice', 'cldice', 'cobertura_principal', 'n_componentes', 'hd95_crudo_mm', 'case_id', 'dataset']


## 2. Resumen de medias


In [5]:
resumen = pd.DataFrame([
    {
        "dataset": "200 casos",
        "n_casos": len(df_200_v1),
        "dice_mean": df_200_v1["dice"].mean(),
        "cldice_mean": df_200_v1["cldice"].mean(),
        "n_componentes_mean": df_200_v1["n_componentes"].mean(),
        "hd95_mean": df_200_v1["hd95_crudo_mm"].mean(),
    },
    {
        "dataset": "1000 casos",
        "n_casos": len(df_1000),
        "dice_mean": df_1000["dice"].mean(),
        "cldice_mean": df_1000["cldice"].mean(),
        "n_componentes_mean": df_1000["n_componentes"].mean(),
        "hd95_mean": df_1000["hd95_crudo_mm"].mean(),
    },
])
print(resumen.to_string(index=False))

   dataset  n_casos  dice_mean  cldice_mean  n_componentes_mean  hd95_mean
 200 casos       40   0.754910     0.822100              14.725  21.063610
1000 casos      200   0.747547     0.811284              16.985  24.719474


## 3. Tests para muestras independientes (Mann-Whitney U y t-test independiente)

El t-test se aplica con `equal_var=False` (Welch), dado que los tamaños muestrales son muy distintos (40 frente a ≈ 200) y no hay motivo para asumir varianzas iguales entre ambos conjuntos de validación.


In [6]:
METRICAS = {
    "dice": "dice",
    "cldice": "cldice",
    "n_componentes": "n_componentes",
    "hd95_crudo_mm": "hd95_crudo_mm",
}

resultados = []
for nombre_metrica, columna in METRICAS.items():
    grupo_200 = df_200_v1[columna].dropna()
    grupo_1000 = df_1000[columna].dropna()

    u_stat, u_p = stats.mannwhitneyu(grupo_200, grupo_1000, alternative="two-sided")
    t_stat, t_p = stats.ttest_ind(grupo_200, grupo_1000, equal_var=False)

    resultados.append({
        "metrica": nombre_metrica,
        "media_200": grupo_200.mean(),
        "media_1000": grupo_1000.mean(),
        "mannwhitney_p": u_p,
        "ttest_welch_p": t_p,
        "significativo_ambos_p<0.05": (u_p < 0.05) and (t_p < 0.05),
    })

df_tests = pd.DataFrame(resultados).round(4)
df_tests.to_csv(BASE / "tests_independientes_200_vs_1000.csv", index=False)
print(df_tests.to_string(index=False))

      metrica  media_200  media_1000  mannwhitney_p  ttest_welch_p  significativo_ambos_p<0.05
         dice     0.7549      0.7475         0.5754         0.5252                       False
       cldice     0.8221      0.8113         0.5636         0.3921                       False
n_componentes    14.7250     16.9850         0.0414         0.0330                        True
hd95_crudo_mm    21.0636     24.7195         0.2800         0.2349                       False


## 4. Interpretación de la tabla

- **`significativo_ambos_p<0.05` = True:** la diferencia entre entrenar con 200 y con 1000 casos es estadísticamente significativa, es decir, el aumento del conjunto de datos tuvo un efecto medible en esa métrica.
- **`significativo_ambos_p<0.05` = False:** con los datos disponibles no hay evidencia suficiente de que escalar a 1000 casos modificara esa métrica de forma significativa.

Al tratarse de conjuntos de validación de distinto tamaño y composición (pacientes distintos), la comparación responde a si el modelo entrenado con más datos generaliza mejor en su propio conjunto de prueba, y no a si mejora exactamente sobre los mismos 40 pacientes. Esta distinción debe quedar explícita en la redacción de los resultados.
